# Haystack

We talked about LangChain's features, and how to utilise them to build language applications. While LangChain supports quite a lot of different use cases in NLP, we are going to talk about another open-source tool called Haystack that is used in building large-scale search systems. Information retrieval which is an area of focus for Haystack, and is also an area of overlap with LangChain. Haystack also supports prompting to achieve summarization, question-answering, translation, etc.

# What is Haystack?

Haystack is a versatile open-source Python framework that provides developers with a toolkit to create powerful search systems that can efficiently handle large document collections. Whether you're building a search engine for a web application, an e-commerce platform, or a knowledge management system, Haystack makes it easy to integrate advanced search capabilities into your project.

In [ ]:
%%bash

pip install -r ../requirements.txt

In [ ]:
# initialize the DocumentStore
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

In [ ]:
from datasets import load_dataset
from haystack import Document

In [ ]:
# extract and load
dataset = load_dataset("bilgeyucel/seven-wonders", split="train")

In [ ]:
dataset

In [ ]:
for doc in dataset:
  print(doc['content'])

In [ ]:
docs = [Document(content=doc['content'], meta=doc['meta']) for doc in dataset]

In [ ]:
docs

In [ ]:
# initializing the embedding model
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

In [ ]:
# embedding and storing the docs
docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings['documents'])

In [ ]:
# Initialize a Text Embedder for user query
from haystack.components.embedders import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# Initialize the Retriever
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store)

In [ ]:
# creating prompt template
from haystack.components.builders import PromptBuilder

template = """
Given the following information, answer the question.

Content:
{% for document in documents %}
  {{ document.content }}
{% endfor %}

Question: {{question}}
Answer:
"""

prompt_builder = PromptBuilder(template=template)

In [ ]:
import os
from getpass import getpass
from haystack_integrations.components.generators.google_ai import GoogleAIGeminiGenerator

In [ ]:
if "GOOGLE_API_KEY" not in os.environ:
  os.environ['GOOGLE_API_KEY'] = getpass("Enter GOOGLE API KEY:")

In [ ]:
generator = GoogleAIGeminiGenerator(model="gemini-2.5-flash")

In [ ]:
# creating object
from haystack import Pipeline

basic_rag_pipeline = Pipeline()

In [ ]:
# Add components to your pipeline
basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", generator)


In [ ]:
# Now, connect the components to each other
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever", "prompt_builder")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm")

In [ ]:
question = "What does Rhodes Statue look like?"

In [ ]:
response = basic_rag_pipeline.run({"text_embedder": {"text": question}, "prompt_builder": {"question": question}})

In [ ]:
response

In [ ]:
examples = [
    "Where is Gardens of Babylon?",
    "Why did people build Great Pyramid of Giza?",
    "What does Rhodes Statue look like?",
    "Why did people visit the Temple of Artemis?",
    "What is the importance of Colossus of Rhodes?",
    "What happened to the Tomb of Mausolus?",
    "How did Colossus of Rhodes collapse?",
]

In [ ]:
question = "Why did people visit the Temple of Artemis?"

response = basic_rag_pipeline.run({"text_embedder": {"text": question}, "prompt_builder": {"question": question}})
response

In [ ]:
response['llm']['replies'][0]